In [5]:
!pip install langchain langchain-community sentence-transformers faiss-cpu accelerate

In [6]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

print("1. Chargement du fichier CSV...")
# Chargez le fichier que nous avons produit au Devoir 1
try:
    df = pd.read_csv("export_final_arabert.csv")
except FileNotFoundError:
    df = pd.read_csv("export_final.csv")

print(f"Données chargées : {len(df)} infractions trouvées.")

print("\n2. Nettoyage et transformation en Documents LangChain...")
documents =[]
for index, row in df.iterrows():
    # On construit une phrase riche qui sera facile à comprendre pour l'IA
    texte_complet = f"ح حسب المادة {row['article_id']} من مدونة السير: "
    texte_complet += f"المخ المخالفة هي {row['infraction_desc']}. "

    if pd.notna(row['amende_fixe']):
        texte_complet += f"غرامة هذه المخالفة هي {row['amende_fixe']} درهم. "
    if pd.notna(row['points_retrait']):
        texte_complet += f"يتم خصم {row['points_retrait']} نقط. "
    if pd.notna(row['categorie_vehicule']) and row['categorie_vehicule'] != "Non spécifié":
         texte_complet += f"صنف المركبة: {row['categorie_vehicule']}. "

    # On crée l'objet Document en gardant les métadonnées pour retrouver la source plus tard
    doc = Document(
        page_content=texte_complet,
        metadata={"article_id": str(row['article_id']), "mots_cles": str(row['mots_cles'])}
    )
    documents.append(doc)

print("\n3. Découpage en unités exploitables (Chunks)...")
# On configure le découpeur (Chunk size de 300 caractères, avec un chevauchement de 50 pour ne pas couper le sens)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Succès ! {len(documents)} documents ont été transformés en {len(chunks)} chunks.")
print("-" * 60)
print("Aperçu du premier chunk généré :")
print(chunks[0].page_content)

1. Chargement du fichier CSV...
Données chargées : 336 infractions trouvées.

2. Nettoyage et transformation en Documents LangChain...

3. Découpage en unités exploitables (Chunks)...
✅ Succès ! 336 documents ont été transformés en 547 chunks.
------------------------------------------------------------
Aperçu du premier chunk généré :
ح حسب المادة 1 من مدونة السير: المخ المخالفة هي 1 املاده ال يجوز الي شخص ان يسوق مركبه ذات محرك او مجموعه مركبات علا الطريق العموميه ما لم يكن حاصال علا رخصه للسياقه ساريه الصالحيه ومسلمه من قبل االداره، تناسب صنف .املركبه او مجموعه املركبات التي يسوقها. صنف المركبة: leger.


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

print("1. Extraction des textes de nos chunks...")
# On récupère juste le texte (page_content) de chaque chunk
texts = [chunk.page_content for chunk in chunks]

print("2. Chargement du modèle d'Embedding (Multilingue/Arabe)...")
# On utilise un modèle multilingue adapté à l'arabe au lieu de l'anglais basique
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("3. Encodage des textes en vecteurs...")
embeddings = embedding_model.encode(texts)

print("4. Création de l'index FAISS...")
# Dimension des vecteurs générés par notre modèle (généralement 384 pour MiniLM)
dimension = embeddings.shape[1]

# Initialisation de FAISS avec la distance L2 (distance euclidienne)
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print(f"✅ Succès ! Taille de l'index FAISS : {index.ntotal} documents vectorisés.")
print("-" * 60)

# 5. Création de la fonction de recherche (Retriever) comme montrée par le prof
def retrieve(query, k=3):
    """
    Transforme la question en vecteur et cherche les k documents les plus proches dans FAISS.
    """
    # Convertir la question en vecteur
    query_vec = embedding_model.encode([query])

    # Chercher les documents similaires
    distances, indices = index.search(np.array(query_vec), k)

    # FAISS retourne les vecteurs les plus proches et leurs positions
    return [texts[i] for i in indices[0]]

# --- TEST DE NOTRE MOTEUR DE RECHERCHE ---
question_test = "ما هي غرامة التوقف أو الوقوف غير القانوني؟" # (Quelle est l'amende pour stationnement illégal ?)
print(f"Question de test : {question_test}\n")
print("Documents trouvés par FAISS :")
resultats = retrieve(question_test, k=2)
for i, res in enumerate(resultats):
    print(f"\n--- Résultat {i+1} ---")
    print(res)

1. Extraction des textes de nos chunks...
2. Chargement du modèle d'Embedding (Multilingue/Arabe)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3109.31it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Encodage des textes en vecteurs...
4. Création de l'index FAISS...
✅ Succès ! Taille de l'index FAISS : 547 documents vectorisés.
------------------------------------------------------------
Question de test : ما هي غرامة التوقف أو الوقوف غير القانوني؟

Documents trouvés par FAISS :

--- Résultat 1 ---
املذكوره رصيد من من هذا القانون وعند انصرام. صنف المركبة: leger.

--- Résultat 2 ---
بقوه القانون، اذا طلبها املعني باالمر وذلك علا.


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("1. Chargement du modèle de langage (Qwen 0.5B) - Cela peut prendre 1 à 2 minutes...")
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

# Chargement du tokenizer et du modèle
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("2. Création du générateur (Pipeline)...")
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

def rag_answer(query):
    # 1. Recherche des documents pertinents (Retriever)
    docs = retrieve(query, k=3)

    # 2. Construction du contexte (fusion des documents en un seul bloc)
    context = "\n".join(docs)

    # 3. Construction du Prompt (Traduit en arabe pour notre contexte marocain)
    prompt = f"""أنت خبير في قوانين مدونة السير المغربية.
استخدم فقط السياق أدناه للإجابة على السؤال.

السياق:
{context}

السؤال:
{query}

الجواب:
"""

    # 4. Génération de la réponse par le LLM
    output = generator(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3,
        return_full_text=False # Pour n'afficher que la réponse, sans répéter le prompt
    )

    return output[0]["generated_text"].strip()

# --- TEST DU RAG ---
print("\n3. Test du système RAG global...")
question = "ما هي غرامة التوقف غير القانوني؟"
print(f"Question : {question}\n")
print("L'IA réfléchit et rédige sa réponse...")

reponse = rag_answer(question)
print("\n" + "="*50)
print(f"Réponse générée par l'IA :\n{reponse}")
print("="*50)

1. Chargement du modèle de langage (Qwen 0.5B) - Cela peut prendre 1 à 2 minutes...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4328.67it/s]
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


2. Création du générateur (Pipeline)...

3. Test du système RAG global...
Question : ما هي غرامة التوقف غير القانوني؟

L'IA réfléchit et rédige sa réponse...

Réponse générée par l'IA :
102
المست Answered Question:

ما هي الغرامة التوقف غير القانوني؟

الجواب:

102

هذه الردود تلتزم بالنص القانوني والمعترف به في المغرب، حيث أن المادة 102 من مدونة السير المغربية تعطي الغرامة التوقف غير القانوني 102. 

لذا، فإن الغرامة التي يجب أن يتعرض لها المركب إذا لم يتم الالتزام بالقوانين المعمول بها هي 102. 

الجواب الصحيح هو 102. 

من الجدير بالذكر أن هذه الحالة قد تكون غير متوقعة أو غير واضحة في بعض الحالات،


In [9]:
!pip install gradio

   ---------------------------------------- 0.0/19.7 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.7 MB 5.6 MB/s eta 0:00:04
   ------ --------------------------------- 3.1/19.7 MB 9.7 MB/s eta 0:00:02
   ---------- ----------------------------- 5.2/19.7 MB 10.3 MB/s eta 0:00:02
   --------------- ------------------------ 7.9/19.7 MB 10.8 MB/s eta 0:00:02
   -------------------- ------------------- 10.2/19.7 MB 11.0 MB/s eta 0:00:01
   ------------------------ --------------- 12.1/19.7 MB 10.5 MB/s eta 0:00:01
   ----------------------------- ---------- 14.4/19.7 MB 10.7 MB/s eta 0:00:01
   ---------------------------------- ----- 16.8/19.7 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  19.4/19.7 MB 10.9 MB/s eta 0:00:01
   ---------------------------------------  19.7/19.7 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------- 19.7/19.7 MB 9.9 MB/s  0:00:02

   ----------------------------------------  0/11 [pydub]
   --- -----

In [10]:
import gradio as gr

custom_css = """
body {
    background: linear-gradient(135deg, #0f172a, #1e293b);
}

.gradio-container {
    font-family: 'Segoe UI', sans-serif;
}

h1 {
    text-align: center;
    color: white;
    font-size: 34px;
    margin-bottom: 5px;
}

.subtitle {
    text-align:center;
    color:#cbd5e1;
    margin-bottom:25px;
}

textarea {
    border-radius: 12px !important;
    border: 1px solid #334155 !important;
}

button {
    border-radius: 12px !important;
    font-weight: bold !important;
    font-size: 16px !important;
}

#box {
    background: #111827;
    padding: 25px;
    border-radius: 18px;
    box-shadow: 0 0 25px rgba(0,0,0,0.35);
}
"""

def interface_rag(question):
    return rag_answer(question)

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.HTML("<h1>⚖️ Assistant Juridique Routier Marocain</h1>")
    gr.HTML("<div class='subtitle'>Réponses intelligentes basées sur le Code de la Route</div>")

    with gr.Column(elem_id="box"):

        question = gr.Textbox(
            label="Votre question",
            placeholder="Ex: Quelle est l'amende pour excès de vitesse ?",
            lines=3
        )

        output = gr.Textbox(
            label="Réponse",
            lines=10
        )

        with gr.Row():
            btn = gr.Button("🚀 Envoyer", variant="primary")
            clear = gr.Button("🗑️ Effacer")

        btn.click(fn=interface_rag, inputs=question, outputs=output)
        clear.click(lambda: ("", ""), outputs=[question, output])

demo.launch()

C:\Users\Me\AppData\Local\Temp\ipykernel_17748\3797189534.py:47: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
